In [13]:
from collections import defaultdict
import csv

new_music_ids = defaultdict(int)
new_track = []
track_fp = "../data/raw/music/50/track.csv"
prefix = "t"
with open(track_fp, "r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    for i, row in enumerate(reader):
        original_id = row.get("track", "").strip()
        new_id = f"{prefix}{i}"
        row["track"] = new_id
        new_music_ids[original_id] = new_id
        # write 
        new_track.append(row)

with open(f"music_new_ids/track.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(new_track)


In [14]:
# Duplicates
music_dups = "../data/raw/music/50/track_dups.csv"
new_track_dups_fp = "music_new_ids/track_dups.csv"
new_track_dups = []
with open(music_dups, 'r') as infile:
    reader = csv.DictReader(infile)
    for row in reader:
        new_row = {"1": 0, "2": 0}
        original_id = row["1"]
        new_row["1"] = new_music_ids[row["1"]]
        new_row["2"] = new_music_ids[row["2"]]
        new_track_dups.append(new_row)

fieldnames = ["1","2"]
with open(new_track_dups_fp, 'w') as outfile:
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(new_track_dups)

In [15]:
import pandas as pd

def create_junction_table_pandas(csv_file_path: str, pk: str, fk: str, out):
    df = pd.read_csv(csv_file_path)
    junction_df = df[[pk, fk]]
    junction_df.to_csv(out, index=False)

In [16]:
import pandas as pd

def flatten_track_to_artist(track_csv: str, junction_csv: str, output_name: str = "music_new_ids/track_artist_direct.csv"):
    """
    Creates a direct junction between tracks and artists.
    
    track_csv: Contains [track_id, artist_credit]
    junction_csv: The 'artist_credit_name.csv' containing [artist_credit, artist]
    """
    # 1. Load the tracks and the credit-to-artist junction
    # We only need the ID columns to keep memory usage low
    df_tracks = pd.read_csv(track_csv, usecols=['track', 'artist_credit'])
    df_junction = pd.read_csv(junction_csv, usecols=['artist_credit', 'artist'])

    # 2. Merge tracks with the junction table on the credit ID
    # This 'explodes' each track into multiple rows if it has multiple artists
    direct_map = pd.merge(
        df_tracks, 
        df_junction, 
        on='artist_credit'
    )

    # 3. Clean up: Keep only track ID and artist ID
    # Rename columns if your CSVs use different headers (e.g., 'id' vs 'track_id')
    final_junction = direct_map[['track', 'artist']].rename(
        columns={'track': 'track', 'artist': 'artist'}
    )

    # 4. Remove duplicates
    # Necessary if an artist is credited twice on the same track for some reason
    print(final_junction.shape[0])
    final_junction = final_junction.drop_duplicates()
    print(final_junction.shape[0])
    # 5. Export
    final_junction.to_csv(output_name, index=False)
    print(f"Direct junction created: {output_name} ({len(final_junction)} rows)")

# Example Usage:
flatten_track_to_artist(track_fp,"../data/raw/music/50/artist_credit_name.csv")

6519
6420
Direct junction created: music_new_ids/track_artist_direct.csv (6420 rows)


In [17]:
# Related Tables
related_tables = ["track_artist_direct.csv"]

for t in related_tables:
    newname = f"{t}"
    # fp = f"../data/raw/music/50/{t}"
    fp = f"music_new_ids/{t}"
    # fp = t
    new_rows = []
    with open(fp, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        for row in reader:
            row["track"] = new_music_ids[row["track"]]
            new_rows.append(row)

    with open(f"music_new_ids/{newname}", 'w') as outfile:
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(new_rows)



In [18]:
def write_new_ids(entity, in_fp, out_fp, prefix, related_tables):
    new_ids = {}
    new_rows = []
    with open(in_fp, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        for i, row in enumerate(reader):
            original_id = row.get(entity, "").strip()
            row[entity] = f"{prefix}{i}"
            new_ids[original_id] = f"{prefix}{i}"
            # write 
            new_rows.append(row)

    with open(out_fp, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(new_rows)


    dups = f"../data/raw/music/50/{entity}_dups.csv"
    out = f"music_new_ids/{entity}_dups.csv"
    new_rows = []
    with open(dups, 'r') as infile:
        reader = csv.DictReader(infile)
        fieldnames = reader.fieldnames
        for row in reader:
            new_row = {"1": 0, "2": 0}
            original_id = row["1"]
            new_row["1"] = new_ids[row["1"]]
            new_row["2"] = new_ids[row["2"]]
            new_rows.append(new_row)

    with open(out, 'w') as outfile:
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(new_rows)

    for t in related_tables:
        # newname = f"new_{t}"
        # fp = f"../data/raw/music/50/{t}"
        new_rows = []
        with open(f"music_new_ids/{t}", "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            fieldnames = reader.fieldnames
            for row in reader:
                if row[entity]:
                    row[entity] = new_ids[row[entity]]
                    new_rows.append(row)

        with open(f"music_new_ids/{t}", 'w') as outfile:
            writer = csv.DictWriter(outfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(new_rows)

# recording-track junction 
entity = "recording"
in_fp = f"../data/raw/music/50/{entity}.csv"
create_junction_table_pandas(in_fp, "recording", "artist_credit", "music_new_ids/recording_artist.csv")

# rename header of junction table recording_artst (stale header: artist_credit)
df = pd.read_csv('music_new_ids/recording_artist.csv')

# Option A: Rename specific columns
df = df.rename(columns={'artist_credit': 'artist'})

# Save the result
df.to_csv('music_new_ids/recording_artist.csv', index=False)

# artists
entity = "artist"
in_fp = f"../data/raw/music/50/{entity}.csv"
create_junction_table_pandas(in_fp, "artist", "area", "music_new_ids/artist_area.csv")
prefix = "a"
related_tables = ["track_artist_direct.csv", "artist_area.csv", "recording_artist.csv"]

write_new_ids(entity, in_fp, f"music_new_ids/{entity}.csv", prefix, related_tables)


KeyError: 'id-1170816'

In [ ]:
prefix = "r"
related_tables = ["recording_artist.csv"]

write_new_ids(entity, in_fp, f"music_new_ids/{entity}.csv", prefix, related_tables)

In [ ]:
# area
entity = "area"
in_fp = f"../data/raw/music/50/{entity}.csv"
prefix = "b"
related_tables = ["artist_area.csv"]

write_new_ids(entity, in_fp, f"music_new_ids/{entity}.csv", prefix, related_tables)